In [ ]:
import re
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from sklearn.metrics.pairwise import cosine_similarity
import os
import random
from datasets import Dataset
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from scipy.spatial.distance import cosine

In [ ]:
def parse_input_file(file_path):
    anchor, pros, cons = "", [], []
    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()
    for line in lines:
        line = line.strip()
        if line.startswith("Discussion Title:"):
            anchor = line.split("Discussion Title:", 1)[1].strip()
        elif "Pro:" in line:
            clean_line = re.sub(r"\[.*?\]\(.*?\)", "", line.split("Pro:", 1)[1].strip())
            pros.append(clean_line)
        elif "Con:" in line:
            clean_line = re.sub(r"\[.*?\]\(.*?\)", "", line.split("Con:", 1)[1].strip())
            cons.append(clean_line)
    return anchor, pros, cons
folder_path = "./Data" # This is folder path where all the training data is present. Please add your folder path here to work this code.
txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
txt_files = random.sample(txt_files, min(1000, len(txt_files)))
triplet_pairs = []

for file_name in txt_files:
    file_path = os.path.join(folder_path, file_name)
    anchor, pros, cons = parse_input_file(file_path)
    for pro in pros:
        for con in cons:
            triplet_pairs.append({
                "pro": pro,
                "con": con,
                "anchor": anchor
            })

In [ ]:
print(len(triplet_pairs))

In [ ]:
random_triplets = random.sample(triplet_pairs, 100000)

train_examples = [
    InputExample(texts=[item['anchor'], item['pro'], item['con']])
    for item in random_triplets
]
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
train_dataloader = DataLoader(train_examples, shuffle = True, batch_size = 16)

In [ ]:
train_loss = losses.TripletLoss(model=model)

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
os.environ["REPORT_TO"] = "none"

model.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs = 5,
    warmup_steps = 100,
    # use_amp = True,
)
triplet_model = model

In [ ]:
from datasets import load_dataset
valid_dataset = load_dataset("timchen0618/Kialo", split="validation")
test_dataset  = load_dataset("timchen0618/Kialo", split="test")
import pprint
test_data = [
    {
        'anchor': item['question'],
        'pro': item['perspectives'][0],
        'con': item['perspectives'][1],
    }
    for item in test_dataset if item['type'] == 'binary'
]

In [ ]:
# Extract sentences from the test data
claims = [item["anchor"] for item in test_data]
pros = [item["pro"] for item in test_data]
cons = [item["con"] for item in test_data]

In [ ]:
base_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
base_claim_embeddings = base_model.encode(claims, convert_to_numpy = True)
base_pro_embeddings = base_model.encode(pros, convert_to_numpy = True)
base_con_embeddings = base_model.encode(cons, convert_to_numpy = True)

In [ ]:
finetuned_claim_embeddings = model.encode(claims, convert_to_numpy = True)
finetuned_pro_embeddings = model.encode(pros, convert_to_numpy = True)
finetuned_con_embeddings = model.encode(cons, convert_to_numpy = True)

In [ ]:
similarities_pro_fined_tuned = [1 - cosine(claim_emb, pro_emb) for claim_emb, pro_emb in zip(finetuned_claim_embeddings, finetuned_pro_embeddings)]
similarities_con_fined_tuned = [1 - cosine(claim_emb, con_emb) for claim_emb, con_emb in zip(finetuned_claim_embeddings, finetuned_con_embeddings)]

similarities_pro_base = [1 - cosine(claim_emb, pro_emb) for claim_emb, pro_emb in zip(base_claim_embeddings, base_pro_embeddings)]
similarities_con_base = [1 - cosine(claim_emb, con_emb) for claim_emb, con_emb in zip(base_claim_embeddings, base_con_embeddings)]


plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.kdeplot(similarities_pro_base, label='Pro (Base)', fill=True, color="green")
sns.kdeplot(similarities_con_base, label='Con (Base)', fill=True, color="red")
plt.title('Base Model')
plt.xlabel('Cosine Similarity')
plt.legend()

plt.subplot(1, 2, 2)
sns.kdeplot(similarities_pro_fined_tuned, label='Pro (Fine-tuned)', fill=True, color="green")
sns.kdeplot(similarities_con_fined_tuned, label='Con (Fine-tuned)', fill=True, color="red")
plt.title('Fine-tuned Model')
plt.xlabel('Cosine Similarity')
plt.legend()

info_text = f"Epochs: {5}\nBatch Size: {16}\nTraining Pairs: {len(random_triplets)}"

plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))


plt.tight_layout()
plt.show()
